# 08 — Transferring Signals with Functional Maps

**The question:** given a functional map $C$ between shapes $A$ and $B$, how do we transfer a scalar function — or any descriptor — from one shape to the other?

## Intuition

A functional map $C$ encodes the pullback operator $T^*$ in the spectral basis. Once we have $C$, transferring a function is a three-step pipeline:

1. **Analyse**: decompose $f$ on $A$ into spectral coefficients $\mathbf{a}$
2. **Transfer**: multiply by $C$ to get coefficients $\mathbf{b} = C\mathbf{a}$ in $B$'s basis
3. **Synthesise**: reconstruct the transferred function on $B$

This is fully analogous to how a digital equaliser works: transform to frequency domain, apply the filter, transform back.

## Minimal math

Let $f: V_A \to \mathbb{R}$ be a function on shape $A$ stored as a vector $\mathbf{f} \in \mathbb{R}^{n_A}$.

**Step 1 — Project** onto the eigenbasis of $A$:

$$\mathbf{a} = \Phi_A^\top M_A \mathbf{f} \in \mathbb{R}^{K_A}$$

**Step 2 — Transfer** spectral coefficients via $C$:

$$\mathbf{b} = C \mathbf{a} \in \mathbb{R}^{K_B}$$

**Step 3 — Reconstruct** on $B$:

$$\tilde{f} = \Phi_B \mathbf{b} \in \mathbb{R}^{n_B}$$

Putting it together:

$$\tilde{f} = \Phi_B\, C\, \Phi_A^\top M_A \mathbf{f}$$

The quality of the transfer depends on:
- How well $C$ captures the true map (accuracy of the correspondence)
- How many basis functions $K$ we use (spectral resolution)

In [1]:
import gsops.backend as gs
import numpy as np

from geomfum.convert import P2pFromFmConverter
from geomfum.dataset import NotebooksDataset
from geomfum.descriptor.pipeline import (
    ArangeSubsampler,
    DescriptorPipeline,
    L2InnerNormalizer,
)
from geomfum.descriptor.spectral import HeatKernelSignature, WaveKernelSignature
from geomfum.functional_map import (
    FactorSum,
    LBCommutativityEnforcing,
    SpectralDescriptorPreservation,
)
from geomfum.numerics.optimization import ScipyMinimize
from geomfum.plot import MeshPlotter
from geomfum.shape import TriangleMesh

## Setup: load shapes and compute functional map $C$

We reuse the optimisation pipeline from notebook 07.

In [17]:
dataset = NotebooksDataset()
mesh_a = TriangleMesh.from_file(dataset.get_filename("cat-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("lion-00"))

K = 30
mesh_a.laplacian.find_spectrum(spectrum_size=K, set_as_basis=True)
mesh_b.laplacian.find_spectrum(spectrum_size=K, set_as_basis=True)

INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\cat-00.off').
INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\lion-00.off').


(array([-4.34513536e-14,  1.08692580e+01,  1.81320950e+01,  2.90812660e+01,
         3.06173749e+01,  3.12651252e+01,  4.78481503e+01,  8.78504820e+01,
         1.40470146e+02,  1.47856194e+02,  1.49164759e+02,  1.72857180e+02,
         1.76642726e+02,  2.07807791e+02,  2.44858198e+02,  2.76816825e+02,
         3.00135675e+02,  3.01929992e+02,  3.31792544e+02,  3.76725198e+02,
         3.80675117e+02,  4.39189453e+02,  4.40434106e+02,  4.67869854e+02,
         5.01610789e+02,  5.41754796e+02,  5.51533782e+02,  5.68138789e+02,
         5.73241345e+02,  6.27962720e+02]),
 array([[ 1.35986867, -0.40090847, -0.41722911, ...,  1.18429228,
          0.95484369,  0.28436645],
        [ 1.35986867, -0.64177584, -0.52564431, ...,  0.99499392,
         -0.55695433,  1.00448639],
        [ 1.35986867, -0.63871653, -0.53136203, ...,  0.72855765,
         -0.46194355,  0.66082945],
        ...,
        [ 1.35986867, -4.01424037,  8.29737957, ..., -0.07295233,
         -2.81997618, -0.16483989],
   

In [39]:
from geomfum.descriptor.spectral import LandmarkWaveKernelSignature

mesh_a.landmark_indices = gs.array([2840, 1594, 5596, 6809, 3924, 7169])
mesh_b.landmark_indices = gs.array([1334, 834, 4136, 4582, 3666, 4955])

pipeline = DescriptorPipeline(
    [
        HeatKernelSignature.from_registry(n_domain=128),
        LandmarkWaveKernelSignature.from_registry(n_domain=128),
        ArangeSubsampler(subsample_step=4),
        L2InnerNormalizer(),
    ]
)
descr_a = pipeline.apply(mesh_a)
descr_b = pipeline.apply(mesh_b)

objective = FactorSum(
    [
        SpectralDescriptorPreservation(
            mesh_a.basis.project(descr_a),
            mesh_b.basis.project(descr_b),
            weight=1.0,
        ),
        LBCommutativityEnforcing.from_bases(mesh_a.basis, mesh_b.basis, weight=1e-2),
    ]
)
x0 = gs.zeros((K, K))
res = ScipyMinimize(method="L-BFGS-B").minimize(
    objective, x0, fun_jac=objective.gradient
)
C = (res.x).reshape(K, K)
print(f"C computed: {C.shape}")

C computed: (30, 30)


## Transfer helper

In [29]:
def transfer_function(f_a, mesh_a, mesh_b, C):
    """Transfer scalar function f_a from mesh_a to mesh_b using functional map C.

    Steps: project onto A's basis, multiply by C, reconstruct on B's basis.
    """
    # Step 1: spectral coefficients on A
    a = mesh_a.basis.project((f_a))  # (K_A,)
    # Step 2: transfer
    b = C @ a  # (K_B,)
    # Step 3: reconstruct on B
    Phi_B = mesh_b.basis.vecs  # (n_B, K_B)
    return Phi_B @ b  # (n_B,)

## Transfer coordinate functions

The $z$-coordinate on the cat (height) should map to the roughly equivalent height direction on the lion, modulo a possible rotation.

In [30]:
verts_a = mesh_a.vertices
f_z = verts_a[:, 2]  # z-coordinate on the cat

f_z_on_b = transfer_function(f_z, mesh_a, mesh_b, C)

In [31]:
# Source: z-coordinate on cat
plotter = MeshPlotter.from_registry(which="polyscope")
plotter.add_mesh(mesh_a)
plotter.set_vertex_scalars(f_z)
print("Source: z-coordinate on cat")
plotter.show()

Source: z-coordinate on cat


In [32]:
# Transferred to lion
plotter = MeshPlotter.from_registry(which="polyscope")
plotter.add_mesh(mesh_b)
plotter.set_vertex_scalars(f_z_on_b)
print("Transferred: z-coordinate on lion (via functional map)")
plotter.show()

Transferred: z-coordinate on lion (via functional map)


## Transfer an HKS descriptor channel

Spectral descriptors like HKS are intrinsic — a corresponding vertex on lion should have a similar HKS value to the cat. Transferring HKS from cat to lion and comparing with lion's own HKS is a proxy for correspondence quality.

In [33]:
hks = HeatKernelSignature.from_registry(scale=True, n_domain=6)

hks_a = hks(mesh_a)  # (6, n_a)
hks_b = hks(mesh_b)  # (6, n_b)

# Transfer one HKS channel (time scale 0, finest)
channel = 0
hks_a_ch = hks_a[channel]  # (n_a,)
hks_transferred = transfer_function(hks_a_ch, mesh_a, mesh_b, C)  # (n_b,)

In [34]:
# Lion's own HKS (ground truth)
plotter = MeshPlotter.from_registry(which="polyscope")
plotter.add_mesh(mesh_b)
plotter.set_vertex_scalars(hks_b[channel])
print("Lion's own HKS (channel 0)")
plotter.show()

Lion's own HKS (channel 0)


In [35]:
# Transferred HKS from cat via C
plotter = MeshPlotter.from_registry(which="polyscope")
plotter.add_mesh(mesh_b)
plotter.set_vertex_scalars(hks_transferred)
print("Transferred HKS from cat (via functional map)")
plotter.show()

Transferred HKS from cat (via functional map)


## Comparison: functional-map transfer vs pointwise transfer

We can also transfer $f$ via a p2p map extracted from $C$. The two methods agree when $C$ is accurate and $K$ is large enough.

In [36]:
# Convert C to p2p
p2p_converter = P2pFromFmConverter()
p2p = p2p_converter((C), mesh_a.basis, mesh_b.basis)  # (n_b,)

# Direct transfer via p2p indexing
f_z_p2p = f_z[p2p]  # (n_b,)

# Compare the two transfers
diff = np.abs(f_z_on_b - f_z_p2p)
print(f"Max absolute difference (spectral vs p2p transfer): {diff.max():.4f}")
print(f"Mean absolute difference: {diff.mean():.4f}")

Max absolute difference (spectral vs p2p transfer): 0.2890
Mean absolute difference: 0.0501


In [38]:
# Visualise the difference
plotter = MeshPlotter.from_registry(which="polyscope")
plotter.add_mesh(mesh_b)
plotter.set_vertex_scalars(diff)
print("Difference between spectral transfer and p2p transfer")
plotter.show()

Difference between spectral transfer and p2p transfer


## Where to go next

- [09 — Shape Difference Operators](./09_shape_difference_operators.ipynb): use $C$ to quantify how much two shapes differ
- [How to refine a functional map?](../how_to/15_refine_functional_map.ipynb)